# 20 — Inversion engine benchmark

Compares four inversion engines on `S2L2A_example.nc` (267×449, 8 bands)
using `albert_mobley_jax`, 3 free params (C_0, C_Y, C_Mie), identical noise.

| Engine | Loop | Prior | Parallelism | Early exit |
|---|---|---|---|---|
| `lmfit_engine` | Python for-loop | no | sequential px-by-px | per pixel |
| `oe_engine` | static unroll (`n_iter`) | yes | JAX vmap | never |
| `oe_engine_optx` | `lax.while_loop` | yes | JAX vmap | per batch |
| `lsq_engine_optx` | `lax.while_loop` | no | JAX vmap | per batch |

A second sweep varies `tile_size` for `oe_engine_optx` to find the crossover
between per-tile dispatch cost and the one-hard-pixel-holds-the-batch overhead.

In [ ]:
import os
import time
import numpy as np
import xarray as xr
import lmfit
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from bio_optics.water.reflectance import albert_mobley_jax
from bio_optics.inversion import oe_engine, oe_engine_optx, lsq_engine_optx, lmfit_engine
from bio_optics.image_processing import dask_engine

jax.config.update('jax_enable_x64', True)
print('JAX devices:', jax.devices())

In [ ]:
NOISE      = 0.005   # Rrs noise [1/sr]
MAX_STEPS  = 100     # optimistix iteration cap
N_ITER     = 20      # oe_engine static unroll depth
MAX_NFEV   = 200     # lmfit max function evaluations
TILE_SIZE  = 4096    # pixels/tile for engine comparison

DATA_PATH = os.path.join(os.getcwd(), 'example_data', 'S2L2A_example.nc')

# Sentinel-2A band centres for B02-B8A (8 bands)
S2_WAVELENGTHS = np.array([492.4, 559.8, 664.6, 704.1, 740.5, 782.8, 832.8, 864.7])

## 1  Load data

In [ ]:
dataset   = xr.open_dataset(DATA_PATH)
print('Data variables:', list(dataset.data_vars))

band_vars   = list(dataset.data_vars)[1:]   # skip spatial_ref / crs
wavelengths = S2_WAVELENGTHS[:len(band_vars)]

# High-tide (time=1) surface reflectance -> Rrs [1/sr]
Rrs_image = np.stack(
    [dataset.isel(time=1)[v].values for v in band_vars], axis=-1
) / np.pi

n_rows, n_cols, n_obs = Rrs_image.shape
n_water = int(np.isfinite(Rrs_image).all(axis=-1).sum())

print(f'Image   : {n_rows}x{n_cols} = {n_rows*n_cols} pixels, {n_obs} bands')
print(f'n_water : {n_water}  ({100*n_water/(n_rows*n_cols):.1f}% non-NaN)')
print(f'lambda  : {wavelengths[0]:.1f}-{wavelengths[-1]:.1f} nm')

## 2  Parameters & forward model

Four free params in log-space: phytoplankton (C_0), CDOM (C_Y), mineral (C_Mie),
and depth (zB). The example scene is shallow coastal water — fixing zB=100 m would
give zero bottom contribution, making the water-column params unable to fit the
observed elevated red/NIR reflectance and causing non-convergence.

sigma_a=0.7 for all free params (uniform, ~factor-8 range at ±3σ in log-space).
Bottom fractions fixed at f_0=1 (single type); bottom shape from spectral library.

In [ ]:
params = lmfit.Parameters()

# free water-column params
params.add('C_0',   value=0.5,  min=0, max=100, vary=True)
params.add('C_Y',   value=0.1,  min=0, max=10,  vary=True)
params.add('C_Mie', value=0.1,  min=0, max=100, vary=True)

# fixed water-column params
for name in ['C_1','C_2','C_3','C_4','C_5','C_X']:
    params.add(name, value=0.0, vary=False)
params.add('S',                   value=0.014, vary=False)
params.add('S_NAP',               value=0.011, vary=False)
params.add('K',                   value=0.0,   vary=False)
params.add('lambda_0',            value=440.0, vary=False)
params.add('lambda_S',            value=500.0, vary=False)
params.add('T_W',                 value=18.0,  vary=False)
params.add('T_W_0',               value=20.0,  vary=False)
params.add('a_NAP_spec_lambda_0', value=0.041, vary=False)
params.add('bb_phy_spec',         value=0.0010, vary=False)
params.add('bb_Mie_spec',         value=0.0042, vary=False)
params.add('bb_X_spec',           value=0.0086, vary=False)
params.add('n',                   value=-1.0,   vary=False)

# geometry
params.add('theta_sun',  value=np.radians(30), vary=False)
params.add('theta_view', value=np.radians(0),  vary=False)
params.add('n1',         value=1.0,    vary=False)
params.add('n2',         value=1.33,   vary=False)
params.add('kappa_0',    value=1.0546, vary=False)

# bathymetry — zB free; scene is shallow so 100 m fixed would give zero bottom
# contribution and prevent convergence
params.add('zB', value=0.5, min=0.01, max=50.0, vary=True)
for i in range(6):
    params.add(f'f_{i}', value=(1.0 if i == 0 else 0.0), vary=False)
    params.add(f'B_{i}', value=1/np.pi, vary=False)

log_params = ['C_0', 'C_Y', 'C_Mie', 'zB']
sigma_a    = {'C_0': 0.7, 'C_Y': 0.7, 'C_Mie': 0.7, 'zB': 0.7}

free = [n for n, p in params.items() if p.vary]
print(f'Free params ({len(free)}): {free}')

In [ ]:
pre = albert_mobley_jax.precompute(wavelengths)

param_names = list(params.keys())
f_vec = albert_mobley_jax.make_forward_vec(param_names, pre)

# InversionSetup shared by all JAX engines
setup = oe_engine.build_inversion(params, f_vec, sigma_a, log_params=log_params)

x_prior   = jnp.array([float(params[n].value) for n in param_names])
Rrs_prior = np.array(f_vec(x_prior))
print(f'fit_names : {setup.fit_names}')
print(f'Rrs_prior : [{Rrs_prior.min():.4f}, {Rrs_prior.max():.4f}] 1/sr')

In [ ]:
# lmfit_engine needs forward_func(params, wavelengths) -> ndarray
_f_jit = jax.jit(f_vec)

def forward_func(p, wl):
    x = jnp.array([float(p[k].value) for k in param_names])
    return np.array(_f_jit(x))

_ = forward_func(params, wavelengths)   # compile

# method='least_squares' (underscore) = scipy TRF — correct lmfit alias.
# Note: 'least-squares' (hyphen) is NOT a valid alias; it falls back to Nelder-Mead.
setup_lmfit = lmfit_engine.build_inversion(
    params, wavelengths, forward_func,
    method='least_squares', max_nfev=MAX_NFEV,
)
print('lmfit setup done')

## 3  JAX JIT warmup (excluded from benchmark timings)

JIT compile times are shown separately because they reveal a key trade-off:

- **`oe_engine`**: XLA unrolls `n_iter` iterations at compile time → large graph, long compile, but fused execution. Compile cost scales linearly with `n_iter`.
- **`oe_engine_optx` / `lsq_engine_optx`**: `lax.while_loop` compiles only one loop-body iteration → fast compile, but no cross-iteration fusion.

For large images (many tiles) `oe_engine`'s compile cost is amortized. For small images it dominates.

In [ ]:
_prior_rrs = np.array(f_vec(x_prior))
_dummy     = np.tile(_prior_rrs, (TILE_SIZE, 1))

compile_times = {}
for label, fn, kw in [
    ('oe_engine',       oe_engine.invert_image,            dict(n_iter=N_ITER)),
    ('oe_engine_optx',  oe_engine_optx.invert_image_optx,  dict(solver='GaussNewton', max_steps=MAX_STEPS)),
    ('lsq_engine_optx', lsq_engine_optx.invert_image,      dict(max_steps=MAX_STEPS)),
]:
    t0 = time.perf_counter()
    fn(_dummy, setup, NOISE, **kw)
    compile_times[label] = time.perf_counter() - t0
    print(f'{label:<22}  compile: {compile_times[label]:.1f} s')

## 4  Engine comparison

In [ ]:
t0 = time.perf_counter()
res_lmfit = dask_engine.invert_image(
    Rrs_image, setup_lmfit, NOISE,
    invert_fn=lmfit_engine.invert_image,
    tile_size=TILE_SIZE,
)
t_lmfit = time.perf_counter() - t0
chi2_lmfit = float(np.nanmedian(res_lmfit['chi2']))
print(f'lmfit_engine   : {t_lmfit:.1f} s  {1000*t_lmfit/n_water:.2f} ms/px  chi2={chi2_lmfit:.3f}')

In [ ]:
t0 = time.perf_counter()
res_oe = dask_engine.invert_image(
    Rrs_image, setup, NOISE,
    invert_fn=oe_engine.invert_image,
    tile_size=TILE_SIZE,
    n_iter=N_ITER,
)
t_oe = time.perf_counter() - t0
chi2_oe = float(np.nanmedian(res_oe['chi2']))
print(f'oe_engine n={N_ITER} : {t_oe:.1f} s  {1000*t_oe/n_water:.2f} ms/px  chi2={chi2_oe:.3f}')

In [ ]:
t0 = time.perf_counter()
res_oe_optx = dask_engine.invert_image(
    Rrs_image, setup, NOISE,
    invert_fn=oe_engine_optx.invert_image_optx,
    tile_size=TILE_SIZE,
    solver='GaussNewton',
    max_steps=MAX_STEPS,
    store_chi2_spectral=True,
)
t_oe_optx = time.perf_counter() - t0
ns_optx   = res_oe_optx['n_steps']
ns_optx_v = ns_optx[ns_optx >= 0]
chi2_oe_optx = float(np.nanmedian(res_oe_optx['chi2']))
print(f'oe_engine_optx : {t_oe_optx:.1f} s  {1000*t_oe_optx/n_water:.2f} ms/px  '
      f'nstep med={int(np.median(ns_optx_v))} p99={int(np.percentile(ns_optx_v,99))} '
      f'pct@max={100*(ns_optx_v==MAX_STEPS).mean():.1f}%  chi2={chi2_oe_optx:.3f}')

In [ ]:
t0 = time.perf_counter()
res_lsq = dask_engine.invert_image(
    Rrs_image, setup, NOISE,
    invert_fn=lsq_engine_optx.invert_image,
    tile_size=TILE_SIZE,
    max_steps=MAX_STEPS,
    store_chi2_spectral=True,
)
t_lsq = time.perf_counter() - t0
ns_lsq   = res_lsq['n_steps']
ns_lsq_v = ns_lsq[ns_lsq >= 0]
chi2_lsq = float(np.nanmedian(res_lsq['chi2']))
print(f'lsq_engine_optx: {t_lsq:.1f} s  {1000*t_lsq/n_water:.2f} ms/px  '
      f'nstep med={int(np.median(ns_lsq_v))} p99={int(np.percentile(ns_lsq_v,99))} '
      f'pct@max={100*(ns_lsq_v==MAX_STEPS).mean():.1f}%  chi2={chi2_lsq:.3f}')

In [ ]:
rows = [
    ('lmfit_engine',          0.0,      t_lmfit,    '---', '---', '---',  chi2_lmfit),
    (f'oe_engine n={N_ITER}', compile_times.get('oe_engine', 0), t_oe,
                                                    str(N_ITER), str(N_ITER), '0', chi2_oe),
    ('oe_engine_optx GN',     compile_times.get('oe_engine_optx', 0), t_oe_optx,
                                                    str(int(np.median(ns_optx_v))),
                                                    str(int(np.percentile(ns_optx_v,99))),
                                                    f'{100*(ns_optx_v==MAX_STEPS).mean():.1f}%',
                                                    chi2_oe_optx),
    ('lsq_engine_optx LM',    compile_times.get('lsq_engine_optx', 0), t_lsq,
                                                    str(int(np.median(ns_lsq_v))),
                                                    str(int(np.percentile(ns_lsq_v,99))),
                                                    f'{100*(ns_lsq_v==MAX_STEPS).mean():.1f}%',
                                                    chi2_lsq),
]

hdr = f'{"engine":<26} {"compile_s":>10} {"run_s":>7} {"ms/px":>7} {"nst_med":>8} {"p99":>5} {"@max":>6} {"chi2":>7}'
print(hdr)
print('-' * len(hdr))
for eng, tc, t, med, p99, sat, chi2 in rows:
    print(f'{eng:<26} {tc:>10.1f} {t:>7.1f} {1000*t/n_water:>7.2f} {med:>8} {p99:>5} {sat:>6} {chi2:>7.3f}')

## 5  n_steps distribution (dynamic engines)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, ns_arr, label in [
    (axes[0], ns_optx_v,  f'oe_engine_optx  GN  max={MAX_STEPS}'),
    (axes[1], ns_lsq_v,   f'lsq_engine_optx LM  max={MAX_STEPS}'),
]:
    ax.hist(ns_arr, bins=range(0, MAX_STEPS + 2), density=True, color='steelblue', alpha=0.8)
    ax.axvline(MAX_STEPS, color='r', ls='--', lw=1.5, label=f'cap={MAX_STEPS}')
    ax.set_xlabel('n_steps')
    ax.set_ylabel('density')
    ax.set_title(label)
    ax.legend()
plt.tight_layout()
plt.show()

## 6  tile_size sweep — oe_engine_optx

Smaller tiles reduce the one-hard-pixel-holds-the-batch effect at the cost of
more per-tile dispatch overhead and more JIT recompilations (one per distinct size).

In [ ]:
TILE_SIZES = [64, 256, 1024, 4096, 16384, 65536]

# Warm up JIT for each tile size before timing
for ts in TILE_SIZES:
    dummy_ts = np.tile(_prior_rrs, (ts, 1))
    oe_engine_optx.invert_image_optx(
        dummy_ts, setup, NOISE, solver='GaussNewton', max_steps=MAX_STEPS
    )
    print(f'tile_size={ts:>6}  JIT warmed', flush=True)

In [ ]:
sweep = []
for ts in TILE_SIZES:
    t0 = time.perf_counter()
    r = dask_engine.invert_image(
        Rrs_image, setup, NOISE,
        invert_fn=oe_engine_optx.invert_image_optx,
        tile_size=ts,
        solver='GaussNewton',
        max_steps=MAX_STEPS,
    )
    t = time.perf_counter() - t0
    ns_s  = r['n_steps']
    ns_s  = ns_s[ns_s >= 0]
    n_tiles = int(np.ceil(n_rows * n_cols / ts))
    med   = float(np.median(ns_s))
    sat   = float(100 * (ns_s == MAX_STEPS).mean())
    sweep.append(dict(tile_size=ts, wall_s=t, ms_px=1000*t/n_water,
                      n_tiles=n_tiles, n_steps_med=med, pct_at_max=sat))
    print(f'ts={ts:>6}  {t:6.1f}s  {1000*t/n_water:.2f}ms/px  '
          f'tiles={n_tiles:4d}  med={med:.0f}  sat={sat:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ts_list    = [s['tile_size']   for s in sweep]
t_list     = [s['wall_s']      for s in sweep]
ms_list    = [s['ms_px']       for s in sweep]
sat_list   = [s['pct_at_max']  for s in sweep]
med_list   = [s['n_steps_med'] for s in sweep]

axes[0].semilogx(ts_list, t_list, 'o-', color='steelblue')
axes[0].set_xlabel('tile_size [px]')
axes[0].set_ylabel('wall time [s]')
axes[0].set_title('Total wall time')
axes[0].grid(True, which='both', alpha=0.3)

axes[1].semilogx(ts_list, med_list, 's-', color='darkorange', label='median n_steps')
axes[1].semilogx(ts_list, sat_list, 'd--', color='red', label='% at max_steps')
axes[1].set_xlabel('tile_size [px]')
axes[1].set_title('Convergence vs tile_size')
axes[1].legend()
axes[1].grid(True, which='both', alpha=0.3)

axes[2].semilogx(ts_list, ms_list, 'o-', color='forestgreen')
axes[2].set_xlabel('tile_size [px]')
axes[2].set_ylabel('ms / water pixel')
axes[2].set_title('Throughput vs tile_size')
axes[2].grid(True, which='both', alpha=0.3)

plt.suptitle('oe_engine_optx GN — tile_size sweep', fontsize=12)
plt.tight_layout()
plt.show()